## 12. Raw Source Frequency Validation Spot Check

The notebook primarily validates the curated quarterly modelling table. This light spot check reconciles two representative raw inputs against the curated quarterly values using the ETL rule in `src.transform.to_quarterly()` and the source-specific aggregation configured in `src.build_curated_dataset.SeriesSpec`. It is not a full raw-source audit.

In [15]:
from src.transform import to_quarterly
from src.validation import find_date_column

spot_specs = [
    {
        'variable': 'cash_rate',
        'source_family': 'RBA cash rate',
        'source_frequency': 'monthly',
        'path': ROOT / 'dataset' / 'rba' / 'cash_rate_1995_2025.csv',
        'value_col': 'cash_rate',
        'agg': 'mean',
    },
    {
        'variable': 'wti_price',
        'source_family': 'market WTI crude oil',
        'source_frequency': 'daily trading days',
        'path': ROOT / 'dataset' / 'market' / 'wti_crude_oil_1995_2025.csv',
        'value_col': 'Close',
        'agg': 'mean',
    },
]

spot_rows = []
frequency_rows = []
curated_lookup = df_model.set_index('quarter')
for spec in spot_specs:
    raw = pd.read_csv(spec['path'])
    date_col = find_date_column(raw)
    parsed_dates = pd.to_datetime(raw[date_col], errors='coerce')
    parsed_non_null = parsed_dates.dropna().sort_values()
    inferred_frequency = pd.infer_freq(parsed_non_null.head(12)) if len(parsed_non_null) >= 12 else None
    median_spacing_days = parsed_non_null.diff().dt.days.dropna().median() if len(parsed_non_null) > 1 else np.nan
    duplicate_dates = int(parsed_dates.duplicated().sum())

    frequency_rows.append({
        'variable': spec['variable'],
        'source_family': spec['source_family'],
        'source_frequency': spec['source_frequency'],
        'raw_rows': int(len(raw)),
        'date_col': date_col,
        'start_date': parsed_non_null.min().date() if len(parsed_non_null) else None,
        'end_date': parsed_non_null.max().date() if len(parsed_non_null) else None,
        'duplicate_dates': duplicate_dates,
        'inferred_frequency_head': inferred_frequency,
        'median_spacing_days': median_spacing_days,
    })

    quarterly = to_quarterly(raw, value_col=spec['value_col'], output_col=spec['variable'], agg=spec['agg'], date_col=date_col)
    common_quarters = [quarter for quarter in quarterly['quarter'] if quarter in curated_lookup.index and pd.notna(curated_lookup.loc[quarter, spec['variable']])]
    sample_quarters = []
    if common_quarters:
        sample_quarters.append(common_quarters[0])
    if len(common_quarters) > 1:
        sample_quarters.append(common_quarters[len(common_quarters) // 2])
    for quarter in dict.fromkeys(sample_quarters):
        raw_value = float(quarterly.loc[quarterly['quarter'] == quarter, spec['variable']].iloc[0])
        curated_value = float(curated_lookup.loc[quarter, spec['variable']])
        spot_rows.append({
            'variable': spec['variable'],
            'source_frequency': spec['source_frequency'],
            'aggregation_rule': spec['agg'],
            'spot_quarter': quarter,
            'raw_derived_value': raw_value,
            'curated_value': curated_value,
            'absolute_difference': abs(raw_value - curated_value),
            'match': bool(np.isclose(raw_value, curated_value, rtol=1e-8, atol=1e-8)),
        })

raw_frequency_spot_check = pd.DataFrame(frequency_rows)
raw_curated_reconciliation = pd.DataFrame(spot_rows)
display(raw_frequency_spot_check)
display(raw_curated_reconciliation.round({'raw_derived_value': 6, 'curated_value': 6, 'absolute_difference': 10}))

display(Markdown('This spot check verifies representative monthly and daily inputs only. Full source-level validation remains an ETL/data-quality task; this EDA validates the curated quarterly table used for modelling.'))

,variable,source_family,source_frequency,raw_rows,date_col,start_date,end_date,duplicate_dates,inferred_frequency_head,median_spacing_days
0,cash_rate,RBA cash rate,monthly,372,date,1995-01-01,2025-12-01,0,MS,31.0
1,wti_price,market WTI crude oil,daily trading days,6367,Date,2000-08-23,2025-12-31,0,None,1.0


,variable,source_frequency,aggregation_rule,spot_quarter,raw_derived_value,curated_value,absolute_difference,match
0,cash_rate,monthly,mean,1995Q1,7.500000,7.500000,0.0,True
1,cash_rate,monthly,mean,2010Q3,4.500000,4.500000,0.0,True
2,wti_price,daily trading days,mean,2000Q3,33.527407,33.527407,0.0,True
3,wti_price,daily trading days,mean,2013Q2,94.173281,94.173281,0.0,True


This spot check verifies representative monthly and daily inputs only. Full source-level validation remains an ETL/data-quality task; this EDA validates the curated quarterly table used for modelling.